# Structure tensor: which way does it point?

Many materials have a **direction**: fibres in a composite, the threads of a rope, the layers of wood. The **structure tensor** measures that direction at every voxel from the image gradients alone, without segmenting or tracing anything.

**How it works.** At each voxel qim3d does three things:

1. It computes the intensity gradient, after smoothing the image with a Gaussian of width **`sigma`** (the *noise scale*).
2. It averages the outer product of the gradient with itself over a neighbourhood of width **`rho`** (the *integration scale*). This is the structure tensor, a 3×3 matrix.
3. It computes the eigenvalues and eigenvectors of that matrix.

For fibre-like structures, the intensity changes strongly *across* a fibre and hardly at all *along* it. So the **smallest eigenvalue** belongs to the eigenvector pointing along the fibre. `qim3d.processing.structure_tensor` returns exactly this pair:

- `val`: the eigenvalues (smallest first), shape `(3, Z, Y, X)`
- `vec`: the eigenvector of the smallest eigenvalue, the local fibre direction, shape `(3, Z, Y, X)`

**Axis order.** The components of `vec` are ordered **(x, y, z)**, where x is the *last* array axis. So `vec[2]` is the component along array axis 0.

In this notebook we will compute the structure tensor, read it in 2D, check it against geometry we know, learn how `sigma` and `rho` matter, and visualize fibres in 3D with **streamlines**.

In [ ]:
import numpy as np
import qim3d

## Part 1: A test object with a known answer, a twisted rope

`qim3d.generate.rope` builds a rope of twisted threads, together with a label volume that tells us which thread each voxel belongs to. Because we know how the rope was built, we can later check the structure tensor against theory.

In [ ]:
twist_rate = 2.0
rope, thread_labels = qim3d.generate.rope(
    shape=(200, 80, 80), num_threads=18, twist_rate=twist_rate, return_labels=True, seed=42
)
print(f"Rope shape: {rope.shape}")

In [ ]:
qim3d.viz.volumetric(rope)

The labels, shown at five depths along the rope (axis 0), reveal the twist: the pattern of threads rotates as we move along the rope.

In [ ]:
cmap = qim3d.viz.colormaps.segmentation(n_labels=thread_labels.max())
qim3d.viz.slices_grid(thread_labels, slice_positions=[10, 55, 100, 145, 190], colormap=cmap, max_value=thread_labels.max(), image_size=2.5)

## Part 2: Compute the structure tensor

We choose `sigma=1.0` (a little smoothing against noise) and `rho=6.0` (about the width of a thread).

In [ ]:
val, vec = qim3d.processing.structure_tensor(rope, sigma=1.0, rho=6.0)

print(f"val: {val.shape}, vec: {vec.shape}")

The eigenvalues tell us what kind of structure we have. Inside the rope, one eigenvalue should be much smaller than the other two, which is the signature of **line-like** structures (two small eigenvalues would mean sheets, three similar ones isotropic structure or noise):

In [ ]:
inside = rope > 0
for i in range(3):
    print(f"mean eigenvalue {i + 1} (inside rope): {val[i][inside].mean():8.1f}")

## Part 3: Reading the result in 2D

`qim3d.viz.vectors` shows the orientation in a slice. It has three panels:

- **left**: arrows showing the in-plane direction of the fibres,
- **middle**: a histogram of the in-plane angles,
- **right**: the orientation as colour. The **hue** is the in-plane angle (as in the histogram). **Grey** means the fibre points *out of* the slice, and a fully saturated colour means it lies *in* the slice.

We pass `background=0` so the empty space around the rope is ignored. Use the slider to move through the volume.

**Looking along the rope** (`axis=0`): the threads are mostly perpendicular to this slice, so the colours are pale. The hue changes smoothly around the rope. The threads lean in a *swirling* pattern, and the angle histogram is spread over all angles.

In [ ]:
qim3d.viz.vectors(rope, vec, axis=0, background=0, grid_size=4)

**Looking from the side** (`axis=1`): the threads now lie almost in the slice, so the colour is much more saturated, and the in-plane angle is concentrated in a single narrow peak.

In [ ]:
qim3d.viz.vectors(rope, vec, axis=1, background=0, grid_size=4)

## Part 4: Does it match the geometry?

The synthetic rope gives us a prediction. Each thread follows a helix around the rope axis, turning by `twist_rate · π` radians per `length` voxels. A thread at distance `r` from the axis is therefore tilted away from the axis by

$$\theta = \arctan\!\left(r \cdot \frac{\text{twist\_rate}\cdot\pi}{\text{length}}\right)$$

Threads in the centre are straight, and threads further out are more tilted. We measure the tilt of each thread from `vec` as the angle between the fibre direction and the rope axis, `arccos(|vec[2]|)`, and compare. This is plain NumPy on the arrays we already have. We stay away from the two ends of the rope, where the filters see the border.

In [ ]:
length = rope.shape[0]
omega = twist_rate * np.pi / length               # rotation per voxel along the rope (radians)
centre = np.array(rope.shape[1:]) // 2            # position of the rope axis in the (y, x) plane
yy, xx = np.mgrid[0 : rope.shape[1], 0 : rope.shape[2]]
z_range = slice(40, 160)

thread_ids = np.arange(1, thread_labels.max() + 1)

# distance of every thread from the rope axis
r_thread = np.array([
    np.mean([np.hypot(yy[thread_labels[z] == t].mean() - centre[0], xx[thread_labels[z] == t].mean() - centre[1])
             for z in range(40, 160, 10)])
    for t in thread_ids
])
predicted = np.degrees(np.arctan(r_thread * omega))


def measured_tilt(vec):
    # angle between the fibre direction and the rope axis, median per thread
    tilt = np.degrees(np.arccos(np.clip(np.abs(vec[2]), 0, 1)))[z_range]
    labels = thread_labels[z_range]
    return np.array([np.median(tilt[labels == t]) for t in thread_ids])


measured = measured_tilt(vec)

ring = np.round(r_thread / 8).astype(int)        # the threads sit in rings at about r = 0, 8 and 16
print(f"{'ring radius':>11s} {'threads':>8s} {'measured tilt':>14s} {'predicted tilt':>15s}")
for k in np.unique(ring):
    sel = ring == k
    print(f"{r_thread[sel].mean():11.1f} {sel.sum():8d} {measured[sel].mean():13.1f}° {predicted[sel].mean():14.1f}°")
print(f"\nMean absolute error over all threads: {np.abs(measured - predicted).mean():.1f}°")

The structure tensor recovers the twist to within a few degrees. Close to the axis the measured tilt is slightly too large, simply because an angle can never be negative and noise pushes it up.

## Part 5: Choosing `sigma` and `rho`

The two scales should match the structure you look at. We keep `sigma=1` and vary `rho`, scoring each result against the prediction from the previous section:

In [ ]:
print(f"{'rho':>4s} {'mean abs error':>15s}")
for rho in [2, 4, 6, 10, 16]:
    _, vec_r = qim3d.processing.structure_tensor(rope, sigma=1.0, rho=rho)
    print(f"{rho:4d} {np.abs(measured_tilt(vec_r) - predicted).mean():14.1f}°")

The error is smallest for a moderate `rho`:

- **`rho` too small**: the average is taken over too few voxels, so the direction is noisy.
- **`rho` too large**: the window reaches into the neighbouring threads, which point in different directions, so the directions get averaged away.

A good rule of thumb is to keep `sigma` small compared to the structure (just enough to suppress noise) and to set `rho` close to the width of the fibres or the spacing you want to resolve. Increasing `sigma` too much has a similar blurring effect.

## Part 6: Streamlines

Arrows in a 2D slice only show a small part of the picture. `qim3d.viz.streamlines` follows the fibre direction through the whole volume and draws it as 3D fibres, so you can see how they run. It starts a fibre at seed points on a regular grid inside the object and follows `vec` in both directions until the structure stops being fibre-like.

The colours mean the same as in `qim3d.viz.vectors`: **hue** is the in-plane (x–y) direction and **grey** means the fibre points out of the x–y plane (along z).

Important arguments:

- `background_threshold`: voxels below this intensity are ignored as background. We compute it with `qim3d.operations.threshold`.
- `fiber_spacing`: distance between seed points. Smaller means more fibres, but slower and busier.
- `max_fiber_length`: the maximum number of steps per fibre.
- `show_volume`: also draw the volume itself as context.
- `camera_position`: e.g. `"iso"`, `"xy"`, `"xz"` or `"yz"`.

The viewer is interactive: rotate and zoom with the mouse.

In [ ]:
bg = qim3d.operations.threshold(rope, method="otsu")

qim3d.viz.streamlines(rope, vec, val, background_threshold=bg, fiber_spacing=5, max_fiber_length=400, camera_position="iso")

The rope axis points along z, so most fibres are only pale. The hue changes from one side of the rope to the other: the threads lean one way on the left and the opposite way on the right, which is exactly the twist we measured in Part 4.

## Part 7: A second example, a twisted fibre structure

`fibers_150x150x150` is a real fibre volume. First look at the data:

In [ ]:
fibers = qim3d.examples.fibers_150x150x150
qim3d.viz.slices_grid(fibers, n_slices=3, slice_axis=1, image_size=3)

Going through the volume along axis 1 (y), the fibre direction rotates from nearly vertical to horizontal and back. Let's compute the structure tensor and look at the plane that contains the fibres:

In [ ]:
val_f, vec_f = qim3d.processing.structure_tensor(fibers, sigma=1.0, rho=6.0)
qim3d.viz.vectors(fibers, vec_f, axis=1, grid_size=6)

Now the streamlines. The fibres lie in the x–z plane, i.e. they point out of the x–y plane that the colours refer to, so the colours say little:

In [ ]:
bg_f = qim3d.operations.threshold(fibers, method="otsu")

qim3d.viz.streamlines(fibers, vec_f, val_f, background_threshold=bg_f, fiber_spacing=8, max_fiber_length=300, camera_position="iso")

### Choose the orientation so the colours carry information

The colour scheme works best for fibres lying in the x–y plane. If your fibres lie in another plane, re-orient the volume with `np.transpose`. Here we swap axes 0 and 1, so the original y axis (which the fibres do not use) becomes the new z axis, and the fibres lie in the new x–y plane.

Note that the structure tensor must then be **recomputed** on the re-oriented volume. Transposing `vec` would not be enough, because its components change meaning too.

In [ ]:
fibers_t = np.transpose(fibers, (1, 0, 2))
val_t, vec_t = qim3d.processing.structure_tensor(fibers_t, sigma=1.0, rho=6.0)

qim3d.viz.streamlines(fibers_t, vec_t, val_t, background_threshold=bg_f, fiber_spacing=8, max_fiber_length=300, camera_position="iso")

The full range of colours now shows how the fibre direction changes through the volume: a continuous rotation, layer by layer.

## Summary

| Task | qim3d function |
|---|---|
| Compute orientation at every voxel | `qim3d.processing.structure_tensor(volume, sigma, rho)` |
| Inspect orientation in 2D slices | `qim3d.viz.vectors(volume, vec, axis=...)` |
| Show fibres in 3D | `qim3d.viz.streamlines(volume, vec, val, background_threshold=...)` |

Remember:

- `vec[0], vec[1], vec[2]` are the **x, y, z** components.
- `sigma` suppresses noise, `rho` sets the size of the neighbourhood that is averaged. Match them to your structures.
- Pass `visualize=True` to `structure_tensor` for a quick look straight away.
- For streamlines, orient the volume so that the fibres lie in the x–y plane.